
# 📘 07_Results_Views — Dynamic DQ Views from Metadata

This notebook builds **two unified result views** that automatically combine
metrics and details from all Lakehouse Monitoring profile tables — driven entirely by metadata.

**Views created:**
- `${catalog}.${out_schema}.dq_all_metrics`
- `${catalog}.${out_schema}.dq_all_metric_details`

They are used later for dashboards and scorecards.

---
### Concept
| Input Metadata Table | Purpose |
|----------------------|----------|
| `monitors_control` | Lists which tables are monitored and where their metric tables live |
| `metric_bindings` | Binds metric templates to concrete tables |
| `metric_templates` | Defines reusable metric expressions, thresholds, and DQ dimensions |

When this notebook runs:
1. It detects all enabled table/metric pairs.
2. It joins ratio and detail metrics dynamically.
3. It creates one unified “metrics” view and one “details” view.

In [0]:
# 1️⃣ Setup widgets and parameters

dbutils.widgets.text("catalog",      "dbdemos_steventan",                 "Catalog")
dbutils.widgets.text("admin_schema", "monitoring_admin",                  "Admin Schema")
dbutils.widgets.text("out_schema",   "lakehouse_monitoring_demo_results", "Output Schema (for views)")
dbutils.widgets.text("tz",           "Asia/Singapore",                    "Display Timezone")

catalog      = dbutils.widgets.get("catalog").strip()
admin_schema = dbutils.widgets.get("admin_schema").strip()
out_schema   = dbutils.widgets.get("out_schema").strip()
tz           = dbutils.widgets.get("tz").strip()

print(f"Using: catalog={catalog}, admin_schema={admin_schema}, out_schema={out_schema}, tz={tz}")


## 2️⃣ Read metadata and prepare lookups

We’ll extract:
- `output_schema_name` for each monitored table (from `monitors_control`)
- metric bindings (from `metric_bindings`)
- threshold and dimension info (from `metric_templates`)

In [0]:
from pyspark.sql import functions as F

def esc_str(s: str) -> str:
    return s.replace("'", "''") if s else s

def qcol(name: str) -> str:
    return f"`{name}`"

def qident(fqn: str) -> str:
    return ".".join(f"`{p}`" for p in fqn.split("."))

# --- read metadata
mc = (
    spark.table(f"{catalog}.{admin_schema}.monitors_control")
         .filter("enabled = true")
         .select("table_catalog","table_schema","table_name","output_schema_name")
)
mb = (
    spark.table(f"{catalog}.{admin_schema}.metric_bindings")
         .filter("enabled = true")
         .select("table_catalog","table_schema","table_name","metric_name","template_name")
)
tmpl = (
    spark.table(f"{catalog}.{admin_schema}.metric_templates")
         .select("template_name","description","dimension","threshold_direction",
                 "good_threshold","acceptable_threshold")
)

out_schema_map = {
    (r.table_catalog, r.table_schema, r.table_name): r.output_schema_name
    for r in mc.collect()
}
tmpl_map = {r.template_name: r.asDict() for r in tmpl.collect()}

ratios, details = {}, {}
for b in mb.collect():
    key = (b.table_catalog, b.table_schema, b.table_name)
    if b.metric_name.endswith("_ratio"):
        ratios.setdefault(key, {})[b.metric_name[:-6]] = (b.metric_name, b.template_name)
    elif b.metric_name.endswith("_details_json"):
        details.setdefault(key, {})[b.metric_name[:-13]] = (b.metric_name, b.template_name)

pairs = []
for key, rmap in ratios.items():
    for base, (ratio_metric, ratio_tmpl) in rmap.items():
        if base in details.get(key, {}):
            detail_metric, _ = details[key][base]
            pairs.append((key, base, ratio_metric, detail_metric, ratio_tmpl))

print(f"Discovered {len(pairs)} metric pairs across metadata.")


## 3️⃣ Create or replace view — `dq_all_metrics`

Each record represents a *table × metric × window* combination.  
Includes DQ dimensions, thresholds, and descriptions from metadata.

In [0]:
union_parts = []
for ((tc, ts, tn), base, ratio_metric, detail_metric, tmpl_name) in pairs:
    out_sch = out_schema_map.get((tc, ts, tn))
    if not out_sch:
        continue

    metrics_tbl = f"{out_sch}.{tn}_profile_metrics"
    meta = tmpl_map.get(tmpl_name, {})

    dim_sql  = f"'{esc_str(meta.get('dimension'))}'"             if meta.get("dimension") else "NULL"
    dir_sql  = f"'{esc_str(meta.get('threshold_direction'))}'"   if meta.get("threshold_direction") else "NULL"
    good_sql = str(meta.get("good_threshold"))                   if meta.get("good_threshold") else "NULL"
    acc_sql  = str(meta.get("acceptable_threshold"))             if meta.get("acceptable_threshold") else "NULL"
    desc_sql = f"'{esc_str(meta.get('description'))}'"           if meta.get("description") else "NULL"

    part = f"""
      SELECT
        '{tc}' AS base_catalog, '{ts}' AS base_schema, '{tn}' AS base_table,
        '{base}' AS metric_base,
        {dim_sql} AS dimension,
        {dir_sql} AS threshold_direction,
        {good_sql} AS good_threshold,
        {acc_sql} AS acceptable_threshold,
        {desc_sql} AS description,
        TRY_CAST({qcol(ratio_metric)} AS DOUBLE) AS metric_ratio,
        CAST({qcol(detail_metric)} AS STRING) AS metric_details_json,
        window.start AS window_start,
        window.end AS window_end,
        granularity,
        CAST(from_utc_timestamp(window.start, '{tz}') AS DATE) AS dt
      FROM {qident(metrics_tbl)}
      WHERE column_name=':table'
        AND ({qcol(ratio_metric)} IS NOT NULL OR {qcol(detail_metric)} IS NOT NULL)
    """
    union_parts.append(part.strip())

union_sql = "\nUNION ALL\n".join(union_parts) if union_parts else """
SELECT NULL AS base_catalog,NULL AS base_schema,NULL AS base_table,
NULL AS metric_base,NULL AS dimension,NULL AS threshold_direction,
NULL AS good_threshold,NULL AS acceptable_threshold,NULL AS description,
NULL AS metric_ratio,NULL AS metric_details_json,
NULL AS window_start,NULL AS window_end,NULL AS granularity,NULL AS dt WHERE 1=0
"""

dq_all_metrics_fqn = f"{catalog}.{out_schema}.dq_all_metrics"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{out_schema}")
spark.sql(f"DROP VIEW IF EXISTS {qident(dq_all_metrics_fqn)}")
spark.sql(f"CREATE VIEW {qident(dq_all_metrics_fqn)} AS {union_sql}")
print(f"✅ Created/updated view: {dq_all_metrics_fqn}")


## 4️⃣ Create or replace view — `dq_all_metric_details`

Explodes the JSON details array into rows, exposing `pk`, `reason`, and any extra attributes.

In [0]:
dq_all_metric_details_fqn = f"{catalog}.{out_schema}.dq_all_metric_details"
spark.sql(f"DROP VIEW IF EXISTS {qident(dq_all_metric_details_fqn)}")

details_sql = f"""
CREATE VIEW {qident(dq_all_metric_details_fqn)} AS
WITH arr AS (
  SELECT
    base_catalog, base_schema, base_table, metric_base,
    dimension, threshold_direction, good_threshold, acceptable_threshold, description,
    window_start, window_end, granularity, dt,
    FROM_JSON(metric_details_json, 'array<string>') AS details_arr
  FROM {qident(dq_all_metrics_fqn)}
  WHERE metric_details_json IS NOT NULL AND metric_details_json <> '[]'
),
exploded AS (
  SELECT *, posexplode_outer(details_arr) AS (detail_index, detail_json_str) FROM arr
),
parsed AS (
  SELECT *, FROM_JSON(detail_json_str, 'map<string,string>') AS detail_map FROM exploded
)
SELECT
  base_catalog, base_schema, base_table, metric_base,
  dimension, threshold_direction, good_threshold, acceptable_threshold, description,
  window_start, window_end, granularity, dt, detail_index,
  detail_map['pk'] AS pk, detail_map['reason'] AS reason,
  map_filter(detail_map, (k,v) -> k NOT IN ('pk','reason')) AS detail_attrs
FROM parsed
WHERE detail_map['pk'] IS NOT NULL AND detail_map['reason'] IS NOT NULL
"""
spark.sql(details_sql)
print(f"✅ Created/updated view: {dq_all_metric_details_fqn}")


## 5️⃣ Quick verification

You can preview both views to ensure they’ve been created successfully.

In [0]:
display(spark.sql(f"SELECT * FROM {qident(dq_all_metrics_fqn)} ORDER BY window_start DESC LIMIT 10"))
display(spark.sql(f"SELECT * FROM {qident(dq_all_metric_details_fqn)} ORDER BY window_start DESC, detail_index LIMIT 10"))


## ✅ Summary

You now have two metadata-driven dynamic views:

| View | Description |
|------|--------------|
| **`dq_all_metrics`** | Aggregated metric ratios and metadata across all monitored tables |
| **`dq_all_metric_details`** | Exploded offending-row details with `pk` / `reason` for root-cause analysis |

These can feed into Power BI, Databricks SQL, or any visualization layer to build a unified **Data Quality Scorecard**.